In [ ]:
!pip install --break-system-packages evaluate trackio 'transformers[torch]' tiktoken sentencepiece 'accelerate>=1.1.0'

In [ ]:
import os
import json
import zipfile
import requests
import numpy as np

import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
import trackio

# Download X-Stance

In [ ]:
dataset_url = "https://github.com/ZurichNLP/xstance/raw/refs/heads/master/data/xstance-data-v1.0.zip"

zipfile_path = "xstance-data-v1.0.zip"
extract_dir = "data"

if not os.path.exists(zipfile_path):
    response = requests.get(dataset_url)
    response.raise_for_status()

    with open(zipfile_path, "wb") as f:
        f.write(response.content)

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zipfile_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

# Load JSONL files

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


train_data = load_jsonl(os.path.join(extract_dir, "train.jsonl"))
valid_data = load_jsonl(os.path.join(extract_dir, "valid.jsonl"))
test_data = load_jsonl(os.path.join(extract_dir, "test.jsonl"))

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(valid_data),
    "test": Dataset.from_list(test_data),
})

# Label mapping

In [ ]:
dataset['train'].to_pandas()

In [ ]:
print(dataset["train"].features)
print(dataset["train"][0])

label_names = sorted(set(dataset["train"]["label"]))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", label2id)


def encode_labels(example):
    example["labels"] = label2id[example["label"]]
    return example


dataset = dataset.map(encode_labels)

# Tokenizer

In [ ]:
model_card = "microsoft/mdeberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_card)


def tokenize_fn(batch):
    return tokenizer(
        batch["question"],
        batch["comment"],
        truncation="only_second",
        max_length=512,
        padding=False,
    )


tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=[
        "question",
        "comment",
        "label",
        "numerical_label",
        "author",
        "topic",
        "language",
        "id",
        "question_id",
    ],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model

In [ ]:
num_labels = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    model_card,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,
)
model = model.float()

# Metrics

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1.compute(
        predictions=preds,
        references=labels,
        average="macro",
    )["f1"]

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
    }

# Training config

In [ ]:
BATCH_SIZE = 16
training_args = TrainingArguments(
    output_dir="./mdeberta-v3-xstance",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=850,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    push_to_hub=True,
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    hub_model_id="MatteoFasulo/mdeberta-v3-xstance",
    report_to="trackio",
    seed=42,
    data_seed=42,
)

# Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Train

In [ ]:
trainer.train()

## Results

In [ ]:
trackio.init(project="huggingface")

In [ ]:
valid_results = trainer.evaluate(tokenized_dataset["validation"])
print("Validation results:", valid_results)

test_results = trainer.evaluate(tokenized_dataset["test"])
print("Test results:", test_results)

In [ ]:
trainer.push_to_hub()
model.push_to_hub("MatteoFasulo/mdeberta-v3-xstance")
tokenizer.push_to_hub("MatteoFasulo/mdeberta-v3-xstance")